In [6]:
# CBAM-ResNet50 Training Notebook
# --------------------------------
# This notebook:
# 1. Organizes CUB-200-2011 into train/val/test splits
# 2. Defines CBAM blocks and builds ResNet50-CBAM
# 3. Trains with strong augmentations, label-smoothing, AdamW, ReduceLROnPlateau, gradient-clipping, early-stopping
# 4. Evaluates on test set and plots metrics

# %%
# --- 1. CONFIGURATION ---
import os, random, shutil, time, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights, ResNet
from torchvision.models.resnet import Bottleneck
from torchmetrics.classification import MulticlassAccuracy
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Paths
DATA_ROOT    = Path(r'D:/DS_542_project/CUB_200_2011')  # <- adjust if needed
ORG_DIR      = DATA_ROOT / 'organized'
TRAIN_DIR    = ORG_DIR / 'train'
VAL_DIR      = ORG_DIR / 'val'
TEST_DIR     = ORG_DIR / 'test'

# Hyper-params
NUM_CLASSES    = 200
BATCH_SIZE     = 32
NUM_EPOCHS     = 20
INIT_LR        = 1e-4
WEIGHT_DECAY   = 1e-2
EARLY_PATIENCE = 7
CLIP_NORM      = 5.0
VAL_SPLIT_PCT  = 0.10
NUM_WORKERS    = 4

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Create validation split if missing
if not VAL_DIR.exists():
    print('Creating validation split...')
    random.seed(42)
    for cls in TRAIN_DIR.iterdir():
        imgs = list(cls.glob('*'))
        n_val = max(1, int(VAL_SPLIT_PCT * len(imgs)))
        sampled = random.sample(imgs, n_val)
        for img in sampled:
            dst = VAL_DIR / cls.name
            dst.mkdir(parents=True, exist_ok=True)
            shutil.move(str(img), dst / img.name)
    print('Validation split created.')
else:
    print('Validation split exists, skipping.')

# %%
# --- 2. DATA LOADERS ---
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_tfms = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.1),
    T.ToTensor(),
    T.RandomErasing(p=0.3, scale=(0.02, 0.15)),
    T.Normalize(mean, std),
])

test_tfms = T.Compose([
    T.Resize(256), T.CenterCrop(224),
    T.ToTensor(), T.Normalize(mean, std),
])

train_ds = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = torchvision.datasets.ImageFolder(VAL_DIR,   transform=test_tfms)
test_ds  = torchvision.datasets.ImageFolder(TEST_DIR,  transform=test_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=device.type=='cuda')
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=device.type=='cuda')
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=device.type=='cuda')

print(f'{len(train_ds)} train | {len(val_ds)} val | {len(test_ds)} test')

# %%
# --- 3. CBAM MODULES & MODEL ---
class ChannelAttention(nn.Module):
    def __init__(self, planes, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(planes, planes//ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(planes//ratio, planes, 1, bias=False)
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.mlp(self.avg(x)) + self.mlp(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2,1,k, padding=k//2, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.conv(torch.cat([x.mean(1,True), x.max(1,True).values],1)))

class CBAM(nn.Module):
    def __init__(self, planes):
        super().__init__()
        self.ca = ChannelAttention(planes)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(x * self.ca(x))

class BottleneckCBAM(Bottleneck):
    def __init__(self, *args, **kw):
        super().__init__(*args, **kw)
        self.cbam = CBAM(self.conv3.out_channels)
    def forward(self, x):
        id = x
        out = self.conv1(x); out = self.bn1(out); out = self.relu(out)
        out = self.conv2(out); out = self.bn2(out); out = self.relu(out)
        out = self.conv3(out); out = self.bn3(out)
        out = self.cbam(out)
        if self.downsample is not None:
            id = self.downsample(x)
        return self.relu(out + id)

# (rest of notebook unchanged...)


Device: cuda
Validation split exists, skipping.
5994 train | 594 val | 5794 test
